# prepare_phase3_cpu
Phase 3 CPU Track Stage A1: authenticate Phase 1 & Phase 2 bindings, run `storage_probe`, pre-create `PHASE1_ROOT/phase3/` parent directories, run `E40`/`E41`/`E12` metadata & `en->en` preflight checks, and publish immutable `phase3_cpu_plan`.

- Run on Colab CPU (`requirements-stage2-cpu.txt`, Python 3.11/3.12, single BLAS/OpenMP thread).
- Edit only Cell 3 if `PHASE1_ROOT` differs from `/content/drive/MyDrive/phase1`.
- After completion, flush & unmount Drive before launching parallel Stage A2 CPU runtimes.


In [ ]:
import os, sys, subprocess
from pathlib import Path

if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use Python 3.11/3.12 with the pinned CPU requirements.')
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'

from google.colab import drive
drive.mount('/content/drive')

REPO = Path('/content/Research-phase3-cpu')
CODE_REF = 'phase3'
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Drazr/Research.git', str(REPO)], check=True)
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean checkout; preserve local work before continuing.')
subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO, check=True)
CODE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'FETCH_HEAD'], cwd=REPO, text=True).strip()
subprocess.run(['git', 'checkout', '--detach', CODE_COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-stage2-cpu.txt')], check=True)
print('Fixed code commit:', CODE_COMMIT)
print('Restart if installation requests it, then rerun setup before continuing.')


In [ ]:
# Visible Stage A1 settings.
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')


In [ ]:
import json

if subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() != CODE_COMMIT:
    raise RuntimeError('Checkout changed after setup.')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean Phase 3 checkout.')
if not PHASE1_ROOT.is_dir():
    raise RuntimeError(f'Missing mounted PHASE1_ROOT directory: {PHASE1_ROOT}')

cmd = [
    sys.executable, '-u', '-m', 'src.phase3.cpu_plan', 'prepare',
    '--phase1-root', str(PHASE1_ROOT),
    '--storage-scope', 'persistent-drive',
]
proc = subprocess.Popen(cmd, cwd=REPO, env=os.environ.copy(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
blockers = []
last_complete = None
for line in proc.stdout:
    print(line, end='', flush=True)
    try:
        payload = json.loads(line)
        if isinstance(payload, dict):
            if payload.get('status') == 'blocked':
                blockers.extend(payload.get('blockers', []))
            if payload.get('event') == 'complete':
                last_complete = payload
    except json.JSONDecodeError:
        pass
code = proc.wait()
if code != 0 or last_complete is None:
    detail = '; '.join(blockers) if blockers else f'exit code {code}'
    raise RuntimeError(f'Phase 3 Stage A1 failed ({code}): {detail}')

print('\n=== AUTHORITATIVE PHASE 3 CPU PLAN REFERENCE ===')
print(json.dumps(last_complete['plan_ref'], indent=2))
print('REMINDER: Run drive.flush_and_unmount() before launching parallel Stage A2 runtimes.')
